# TFG — Análisis comparativo de métodos de optimización numérica para la recuperación de parámetros en un modelo de producción agrícola
## Aplicación a parcelas de olivar en el Alentejo (Portugal)
### Samuel Moreno Reaño — 2026

---

## Sobre el objetivo de este cuaderno

Este cuaderno implementa la parte computacional del TFG (**versión 7 — definitiva**).

El diseño adoptado es un **experimento de verificación de consistencia**: la producción
simulada se genera **sin perturbación estocástica** a partir de los parámetros de referencia
calibrados por Rodríguez Sousa et al. (2019, Tesis doctoral) para olivares ibéricos:

$$c_1^* = 0{,}7388, \qquad c_2^* = -0{,}3471, \qquad c_3^* = 0{,}0401$$

Con datos exactos, el ECM en los parámetros verdaderos es exactamente cero. Cualquier
algoritmo correcto debe recuperar $(c_1^*, c_2^*, c_3^*)$ con error numérico despreciable.

Los valores de $P_i$ proceden de la **Tabla 9 de Rodríguez Sousa et al. (2019)**,
trasladados según la gestión equivalente:

| Tipología Alentejo | Gestión equivalente (Estepa) | $P_i$ (kg/ha) |
|-------------------|------------------------------|--------------|
| Ext C, Ext I | Integrada | 3 500 |
| Ext O, Iv O, SIV O | Ecológica | 3 500 |
| Iv C, Iv I | Intensiva | 5 000 |
| SIV I | Altamente intensiva | 10 000 |

---

## El modelo de producción

$$\text{Production}(t) = P_i \cdot \left[ c_1 + c_2 \cdot \ln(W_j - Er_j \cdot t)
    + c_3 \cdot \left(\ln(W_j - Er_j \cdot t)\right)^2 \right]$$

**Linealidad encubierta:** definiendo $u_k = \ln(W_{j(k)} - Er_{j(k)}\,t_k)$, el modelo
es **lineal** en $(c_1, c_2, c_3)$.

---

## Historial de versiones

| Versión | Cambio principal |
|---------|-----------------|
| v1 | Tautología: P_obs = P_exacta sin ruido. ECM = 0 trivial. Solo Hoja1. |
| v2 | Corrección parcial Hoja2. Tautología mantenida. |
| v3 | Ruido gaussiano 5%. Solo t=1: modelo débilmente identificado. |
| v4 | t ∈ {1,20,50,100,150}; BFGS → L-BFGS-B. |
| v5f | Erj siempre desde Hoja3 (valores numéricos explícitos). |
| **v7** | **Parámetros: Rodríguez Sousa (2019). Pi: Tabla 9 tesis. Sin ruido gaussiano.** |


In [ ]:
# ============================================================
# TFG — Análisis comparativo de métodos de optimización numérica
# Aplicación a parcelas de olivar en el Alentejo (Portugal)
# Samuel Moreno Reaño — 2026
# Versión 7 (v7): parámetros de referencia de Rodríguez Sousa (tesis),
#   Pi ajustados por indicación del tutor, sin ruido gaussiano.
# ============================================================


In [ ]:
# ── BLOQUE 0: Importación de librerías ──────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from scipy.optimize import minimize, differential_evolution
import openpyxl
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11, 'legend.fontsize': 9,
})

COLORES_NIVEL = {
    'Minima':   '#1f77b4',
    'Leve':     '#2ca02c',
    'Moderada': '#ff7f0e',
    'Severa':   '#d62728',
}
NIVEL_LABEL = {1: 'Minima', 2: 'Leve', 3: 'Moderada', 4: 'Severa'}


In [ ]:
# ── Producción inicial (kg/ha) por tipología de gestión ────────────────────
# Los valores de Pi no aparecen explícitamente en la literatura disponible
# para las parcelas del Alentejo. Por indicación del tutor, se adoptan los
# siguientes valores ajustados a partir de las producciones de referencia
# de Rodríguez Sousa et al. (2019, Tesis doctoral):
#   · Gestiones extensivas          (Ext C, Ext I, Ext O) : 3 000 kg/ha
#   · Gestiones intensivas          (Iv C, Iv I, Iv O)    : 4 900 kg/ha
#   · Gestiones superintensivas     (SIV I, SIV O)        :10 500 kg/ha
# Producción inicial (kg/ha) por tipología de gestión.
# Fuente: Tabla 9 de Rodríguez Sousa et al. (2019, Tesis doctoral, Cap. 2),
# que recoge los valores de producción anual para las gestiones de referencia
# de la DOP Estepa (Sevilla). Se trasladaron a las tipologías del Alentejo
# según la correspondencia:
#   Ext C / Ext I → integrada → 3 500 kg/ha
#   Ext O / Iv O / SIV O → ecológica → 3 500 kg/ha
#   Iv C / Iv I → intensiva → 5 000 kg/ha
#   SIV I → altamente intensiva → 10 000 kg/ha
PI_MAP = {
    'Ext C': 3500,  'Ext I': 3500,  'Ext O': 3500,
    'Iv C':  5000,  'Iv I':  5000,  'Iv O':  3500,
    'SIV C':10000,  'SIV I':10000,  'SIV O': 3500,
}

NOMBRE_LARGO = {
    'Ext C': 'Extensivo Convencional',
    'Ext I': 'Extensivo Integrado',
    'Ext O': 'Extensivo Organico',
    'Iv C':  'Intensivo Convencional',
    'Iv I':  'Intensivo Integrado',
    'Iv O':  'Intensivo Organico',
    'SIV C': 'Superintensivo Convencional',
    'SIV I': 'Superintensivo Integrado',
    'SIV O': 'Superintensivo Organico',
}


In [ ]:
# ── Parámetros de referencia ────────────────────────────────────────────────
# Valores calibrados por Rodríguez Sousa et al. (2019, Tesis doctoral, Cap. 2)
# para olivares ibéricos en la DOP Estepa (Sevilla).
# c1, c2, c3 son constantes adimensionales ligadas a la pluviometría anual
# y al tipo de suelo.
C1_TRUE, C2_TRUE, C3_TRUE = 0.7388, -0.3471, 0.0401

print("Librerias cargadas correctamente.")
print(f"Parametros de referencia (Rodriguez Sousa et al., 2019):")
print(f"  c1* = {C1_TRUE}   c2* = {C2_TRUE}   c3* = {C3_TRUE}")
print(f"Produccion inicial (Pi) por tipologia de gestion [Tabla 9, Rodriguez Sousa 2019] (kg/ha):")
for k, v in PI_MAP.items():
    print(f"  {k}: {v}")


In [ ]:
# ── BLOQUE 1: Carga de datos ─────────────────────────────────────────────────
# Ajustar rutas si los archivos están en un subdirectorio distinto.
RUTA_TEORICO = "SUELOSTEORICO.xlsx"
RUTA_FQ      = "FQSAMUEL.xlsx"


def leer_hoja_suelo(ruta, sheet):
    """
    Lee una hoja de Suelos_Teorico y devuelve un DataFrame con:
    Gestion, Nivel, DAP (col 2), Sd (col 10), Wj (col 12), OM (col 14).

    Las celdas combinadas de la columna Gestion se rellenan con ffill().
    NO extrae Erj de la hoja — siempre se obtiene de Hoja3.

    Dificultad encontrada: openpyxl devuelve None para las celdas combinadas
    distintas de la primera; es necesario rastrear el último valor no nulo
    de la columna Gestion manualmente.
    """
    wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    ws = wb[sheet]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    records = []
    gestion_actual = None
    for r in rows[1:]:
        if len(r) == 0:
            continue
        if r[0] is not None:
            gestion_actual = r[0]
        if len(r) < 15:
            continue
        nivel = r[1]
        if nivel is None or not isinstance(nivel, (int, float)):
            continue
        dap = r[2]; sd = r[10]; wj = r[12]; om = r[14]
        if dap is None or sd is None or wj is None:
            continue
        records.append({
            'Gestion': str(gestion_actual).strip(),
            'Nivel':   int(nivel),
            'DAP':     float(dap),
            'Sd':      float(sd),
            'Wj':      float(wj),
            'OM':      float(om) if om is not None else np.nan,
        })
    return pd.DataFrame(records)


def leer_hoja3_rusle(ruta):
    """
    Lee Hoja3 (tabla RUSLE) y devuelve DataFrame con Gestion, Nivel, K, C, Erj.

    Los valores de Erj son numéricos explícitos en Hoja3, a diferencia de Hoja2
    donde estaban como fórmulas Excel (no evaluadas por openpyxl).
    """
    mi = {'E': 'Ext', 'I': 'Iv', 'SI': 'SIV', 'e': 'Ext', 'i': 'Iv', 's': 'SIV'}
    mm = {'c': 'C', 'i': 'I', 'o': 'O', 'C': 'C', 'I': 'I', 'O': 'O'}

    wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    ws = wb['Hoja3']
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    records = []
    for r in rows[1:]:
        if len(r) < 6 or r[0] is None or r[1] is None or r[2] is None:
            continue
        intens = str(r[0]).strip()
        manejo = str(r[1]).strip()
        nivel  = r[2]
        k, c, erj = r[3], r[4], r[5]
        if erj is None or not isinstance(erj, (int, float)):
            continue
        g = mi.get(intens, intens) + ' ' + mm.get(manejo, manejo)
        records.append({
            'Gestion': g,
            'Nivel':   int(nivel),
            'K':       float(k),
            'C':       float(c),
            'Erj':     float(erj),
        })
    return pd.DataFrame(records)


# Cargar suelos de Hoja1 y Hoja2
df_h1 = leer_hoja_suelo(RUTA_TEORICO, 'Hoja1')
df_h2 = leer_hoja_suelo(RUTA_TEORICO, 'Hoja2')
df_h2['Gestion'] = df_h2['Gestion'].replace({'IV I': 'Iv I', 'IV O': 'Iv O'})

# Cargar tabla RUSLE (fuente única de Erj)
df_rusle = leer_hoja3_rusle(RUTA_TEORICO)

# Unir ambas hojas de suelo
df_suelo = pd.concat([df_h1, df_h2], ignore_index=True)

# Cruzar con RUSLE para obtener Erj
df_suelo = pd.merge(df_suelo, df_rusle[['Gestion', 'Nivel', 'K', 'C', 'Erj']],
                    on=['Gestion', 'Nivel'], how='left')

# Añadir metadatos
df_suelo['Pi']           = df_suelo['Gestion'].map(PI_MAP)
df_suelo['Nombre_largo'] = df_suelo['Gestion'].map(NOMBRE_LARGO)
df_suelo['Nivel_label']  = df_suelo['Nivel'].map(NIVEL_LABEL)


In [ ]:
# ── Verificaciones críticas ───────────────────────────────────────────────────
n_nan_erj = df_suelo['Erj'].isna().sum()
n_nan_pi  = df_suelo['Pi'].isna().sum()
n_arg_neg = (df_suelo['Wj'] - df_suelo['Erj'] <= 0).sum()

print("=" * 65)
print("VERIFICACION DE DATOS")
print("=" * 65)
print(f"  Total filas cargadas          : {len(df_suelo)}")
print(f"  Filas con Erj nulo            : {n_nan_erj}  (deben ser 0)")
print(f"  Filas con Pi nulo             : {n_nan_pi}   (deben ser 0)")
print(f"  Filas con Wj - Erj <= 0 (t=1): {n_arg_neg}  (deben ser 0)")
print()
print(df_suelo[['Gestion', 'Nivel_label', 'Pi', 'Wj', 'Erj',
                'DAP', 'Sd', 'OM']].to_string(index=False))


In [ ]:
# ── Gráfica de datos de entrada (teóricos) ──────────────────────────────────
df_graf = df_suelo[df_suelo['Erj'] > 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Datos teoricos — Suelos_Teorico.xlsx', fontsize=13)

colores_nivel = [COLORES_NIVEL[NIVEL_LABEL[n]] for n in df_graf['Nivel']]
etiquetas     = [f"{g}({NIVEL_LABEL[n][0]})"
                 for g, n in zip(df_graf['Gestion'], df_graf['Nivel'])]

axes[0].bar(range(len(df_graf)), df_graf['Wj'], color=colores_nivel)
axes[0].set_title('Peso inicial del suelo (Wj)')
axes[0].set_ylabel('Wj (t/ha)')
axes[0].set_xticks(range(len(df_graf)))
axes[0].set_xticklabels(etiquetas, rotation=90, fontsize=6)

axes[1].bar(range(len(df_graf)), df_graf['Erj'], color=colores_nivel)
axes[1].set_title('Tasa de erosion RUSLE (Erj)')
axes[1].set_ylabel('Erj (t/ha/ano)')
axes[1].set_xticks(range(len(df_graf)))
axes[1].set_xticklabels(etiquetas, rotation=90, fontsize=6)

horizonte = df_graf['Wj'] / df_graf['Erj']
axes[2].bar(range(len(df_graf)), horizonte, color=colores_nivel)
axes[2].axhline(100, color='red', linestyle='--', lw=1.2, label='100 anos')
axes[2].set_title('Horizonte de agotamiento (Wj / Erj)')
axes[2].set_ylabel('Anos')
axes[2].set_xticks(range(len(df_graf)))
axes[2].set_xticklabels(etiquetas, rotation=90, fontsize=6)
axes[2].legend()

leyenda = [Patch(facecolor=COLORES_NIVEL[l], label=l)
           for l in ['Minima', 'Leve', 'Moderada', 'Severa']]
fig.legend(handles=leyenda, loc='lower center', ncol=4,
           title='Nivel erosivo', bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('grafica_datos_teorico.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: grafica_datos_teorico.png]")


In [ ]:
# ── BLOQUE 2: Construcción de observaciones sintéticas EXACTAS ──────────────
#
# CAMBIO RESPECTO A VERSIONES ANTERIORES (v1-v5):
# La producción simulada se genera SIN perturbación estocástica (sin ruido
# gaussiano). El objetivo del experimento es verificar la consistencia de los
# métodos de optimización: con datos exactos, cualquier algoritmo correcto debe
# recuperar (c1*, c2*, c3*) con error numérico despreciable y ECM = 0.
# Este diseño — denominado "verificación de recuperación exacta" — es el paso
# previo natural a un experimento con ruido y está plenamente justificado dado
# que los parámetros de referencia proceden de la literatura científica del
# propio modelo (Rodríguez Sousa et al., 2019).
#
TIEMPOS = [1, 20, 50, 100, 150]

df_base = df_suelo[
    (df_suelo['Erj'] > 0) &
    df_suelo[['Wj', 'Erj', 'Pi']].notnull().all(axis=1)
].copy().reset_index(drop=True)

registros = []
for _, fila in df_base.iterrows():
    for t in TIEMPOS:
        x = fila['Wj'] - fila['Erj'] * t
        if x <= 0:
            continue    # suelo agotado: ln no definido
        u       = np.log(x)
        P_exact = fila['Pi'] * (C1_TRUE + C2_TRUE * u + C3_TRUE * u**2)
        if P_exact <= 0:
            continue
        # Sin ruido: P_obs = P_exacta
        registros.append({
            'Gestion':      fila['Gestion'],
            'Nivel':        fila['Nivel'],
            'Nivel_label':  fila['Nivel_label'],
            'Nombre_largo': fila['Nombre_largo'],
            'Pi':           fila['Pi'],
            'Wj':           fila['Wj'],
            'Erj':          fila['Erj'],
            't':            t,
            'ln_x':         u,
            'P_exacta':     P_exact,
            'P_obs':        P_exact,   # <-- exacta, sin ruido
        })

df_fit = pd.DataFrame(registros)

# Vectores para los optimizadores
ln_x_all  = df_fit['ln_x'].values
Pi_all    = df_fit['Pi'].values
P_obs_all = df_fit['P_obs'].values


def produccion(t_arr, Wj, Erj, Pi, c1, c2, c3):
    """Producción estimada para un vector de tiempos. Devuelve NaN al agotar suelo."""
    x = Wj - Erj * t_arr
    x = np.where(x > 0, x, np.nan)
    return Pi * (c1 + c2 * np.log(x) + c3 * np.log(x)**2)


def ecm(params):
    """Error Cuadrático Medio sobre todas las observaciones multi-tiempo."""
    c1, c2, c3 = params
    y_pred = Pi_all * (c1 + c2 * ln_x_all + c3 * ln_x_all**2)
    return np.mean((P_obs_all - y_pred)**2)


# Cota teórica: ECM evaluado en los parámetros verdaderos = 0 (sin ruido)
ecm_verdadero  = ecm([C1_TRUE, C2_TRUE, C3_TRUE])
rmse_verdadero = np.sqrt(ecm_verdadero)

print("=" * 65)
print("EXPERIMENTO SINTETICO (sin ruido)")
print("=" * 65)
print(f"  Observaciones totales          : {len(df_fit)}")
rango_u = ln_x_all.max() - ln_x_all.min()
print(f"  Rango de u_k = ln(Wj-Erj*t)   : [{ln_x_all.min():.2f}, {ln_x_all.max():.2f}]"
      f"  (amplitud {rango_u:.2f})")
print(f"  ECM* (parametros verdaderos)   : {ecm_verdadero:.2e}  (debe ser ~0)")
print(f"  RMSE*                          : {rmse_verdadero:.4f} kg/ha")


In [ ]:
# ── BLOQUE 3: Gráfica de datos de campo (F-Q_Samuel) ────────────────────────
# Esta figura valida cualitativamente la coherencia entre los datos teóricos
# empleados en el experimento y las propiedades edáficas medidas en campo.
# No interviene en la optimización.

try:
    df_fq = pd.read_excel(RUTA_FQ)
    # Columnas esperadas: Gestión, Manejo del suelo, DAP, ...
    col_gest = [c for c in df_fq.columns if 'gest' in c.lower()][0]
    col_dap  = [c for c in df_fq.columns if 'dap' in c.lower() or
                'densidad' in c.lower()][0]
    col_mo   = [c for c in df_fq.columns if 'materia' in c.lower() or
                ' mo' in c.lower() or c.strip().lower() == 'mo'][0]
    df_fq['grupo'] = df_fq[col_gest].apply(
        lambda x: 'Extensivos' if 'xtens' in str(x) else 'Intensivo')

    fig, axes_fq = plt.subplots(1, 2, figsize=(10, 4))
    for grupo, grp in df_fq.groupby('grupo'):
        axes_fq[0].bar(grupo, grp[col_dap].mean(),
                       yerr=grp[col_dap].std(), capsize=5)
        axes_fq[1].bar(grupo, grp[col_mo].mean(),
                       yerr=grp[col_mo].std(), capsize=5)
    axes_fq[0].set_ylabel('DAP (g/cm³)')
    axes_fq[0].set_title('DAP por gestion (F-Q_Samuel)')
    axes_fq[1].set_ylabel('Materia Organica (%)')
    axes_fq[1].set_title('MO por gestion (F-Q_Samuel)')
    fig.suptitle('Propiedades edaficas de campo (F-Q_Samuel.xlsx)', fontsize=12)
    plt.tight_layout()
    plt.savefig('grafica_datos_campo.png', bbox_inches='tight')
    plt.show()
    print("[Insertar en el TFG: grafica_datos_campo.png]")
except Exception as e:
    print(f"[Aviso: no se pudo graficar F-Q_Samuel: {e}]")


In [ ]:
# ── BLOQUE 4: L-BFGS-B con multi-start ──────────────────────────────────────
N_REAL = 100
np.random.seed(0)
puntos = np.random.uniform(-5, 5, size=(N_REAL, 3))
BOUNDS = [(-50, 50)] * 3

resultados_bfgs = []
for x0 in puntos:
    r = minimize(ecm, x0, method='L-BFGS-B', bounds=BOUNDS)
    resultados_bfgs.append({
        'c1': r.x[0], 'c2': r.x[1], 'c3': r.x[2],
        'ECM': r.fun, 'convergio': r.success,
    })

df_bfgs = pd.DataFrame(resultados_bfgs)
mejor_b  = df_bfgs.loc[df_bfgs['ECM'].idxmin()]
c1_b, c2_b, c3_b = mejor_b['c1'], mejor_b['c2'], mejor_b['c3']

print("=" * 65)
print("L-BFGS-B (mejor de 100 realizaciones)")
print("=" * 65)
print(f"  Realizaciones convergentes: {df_bfgs['convergio'].sum()}/{N_REAL}")
print(f"  c1 = {c1_b:+.6f}   (verdadero: {C1_TRUE:+.4f}   error: {abs(c1_b-C1_TRUE):.6f})")
print(f"  c2 = {c2_b:+.6f}   (verdadero: {C2_TRUE:+.4f}   error: {abs(c2_b-C2_TRUE):.6f})")
print(f"  c3 = {c3_b:+.6f}   (verdadero: {C3_TRUE:+.4f}   error: {abs(c3_b-C3_TRUE):.6f})")
print(f"  ECM = {mejor_b['ECM']:.2e}   ECM* = {ecm_verdadero:.2e}")

# Histograma ECM + ajuste en espacio (u, P)
fig, axes_b = plt.subplots(1, 2, figsize=(13, 5))

axes_b[0].hist(df_bfgs['ECM'], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
axes_b[0].axvline(mejor_b['ECM'], color='black', linestyle='--', lw=1.5,
                  label=f"Mejor ECM = {mejor_b['ECM']:.2e}")
axes_b[0].axvline(ecm_verdadero, color='green', linestyle=':', lw=1.5,
                  label=f"ECM* = {ecm_verdadero:.2e}")
axes_b[0].set_xlabel('ECM (kg/ha)²')
axes_b[0].set_ylabel('Frecuencia')
axes_b[0].set_title('Distribucion de ECM — L-BFGS-B (100 realizaciones)')
axes_b[0].legend()

P_pred_b = Pi_all * (c1_b + c2_b * ln_x_all + c3_b * ln_x_all**2)
axes_b[1].scatter(ln_x_all, P_obs_all, label='P_obs (exacta)',
                  color='#555', alpha=0.4, s=25)
axes_b[1].scatter(ln_x_all, P_pred_b, label='Modelo L-BFGS-B',
                  marker='x', color='#e74c3c', s=40)
axes_b[1].set_xlabel('ln(Wj - Erj * t)')
axes_b[1].set_ylabel('Produccion (kg/ha)')
axes_b[1].set_title(f'Ajuste L-BFGS-B (mejor de 100)   ECM={mejor_b["ECM"]:.2e}')
axes_b[1].legend()

plt.tight_layout()
plt.savefig('ajuste_lbfgsb.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: ajuste_lbfgsb.png]")

# Proyecciones a 100 años (L-BFGS-B)
t100        = np.arange(0, 101)
gestiones_u = df_base['Gestion'].unique()
ncols       = 3
nrows       = int(np.ceil(len(gestiones_u) / 3))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
axes = axes.flatten()
for idx, gestion in enumerate(gestiones_u):
    ax = axes[idx]
    for _, fila in df_base[df_base['Gestion'] == gestion].iterrows():
        y = produccion(t100, fila['Wj'], fila['Erj'],
                       fila['Pi'], c1_b, c2_b, c3_b)
        ax.plot(t100, y, 'o-', color=COLORES_NIVEL[fila['Nivel_label']],
                label=fila['Nivel_label'], markersize=2, linewidth=1.5)
    ax.set_title(NOMBRE_LARGO.get(gestion, gestion))
    ax.set_xlabel('Anos')
    ax.set_ylabel('Produccion (kg/ha/ano)')
    hd, lb = ax.get_legend_handles_labels()
    ax.legend(dict(zip(lb, hd)).values(), dict(zip(lb, hd)).keys(), fontsize=8)
for j in range(idx+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Proyeccion de produccion a 100 anos — L-BFGS-B', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('proyeccion_100a_lbfgsb.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: proyeccion_100a_lbfgsb.png]")


In [ ]:
# ── BLOQUE 5: Nelder-Mead ────────────────────────────────────────────────────
casos_nm = {'Caso 1': [1.68, 0.17, -1.01], 'Caso 2': [1.0, 1.0, 1.0]}
res_nm = {}

for nombre, x0 in casos_nm.items():
    r = minimize(ecm, x0, method='Nelder-Mead')
    res_nm[nombre] = r
    print(f"-- {nombre}  (x0 = {x0}):")
    print(f"   c1={r.x[0]:+.6f}  (verdadero: {C1_TRUE:+.4f}  error: {abs(r.x[0]-C1_TRUE):.6f})")
    print(f"   c2={r.x[1]:+.6f}  (verdadero: {C2_TRUE:+.4f}  error: {abs(r.x[1]-C2_TRUE):.6f})")
    print(f"   c3={r.x[2]:+.6f}  (verdadero: {C3_TRUE:+.4f}  error: {abs(r.x[2]-C3_TRUE):.6f})")
    print(f"   ECM={r.fun:.2e}  (ECM*={ecm_verdadero:.2e})")
    print(f"   Convergencia: {'SI' if r.success else 'NO'}")
    print()

fig, axes_nm = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (nombre, r) in zip(axes_nm, res_nm.items()):
    P_pred = Pi_all * (r.x[0] + r.x[1]*ln_x_all + r.x[2]*ln_x_all**2)
    ax.scatter(ln_x_all, P_obs_all, label='P_obs (exacta)',
               color='#555', alpha=0.4, s=25)
    ax.scatter(ln_x_all, P_pred, label='Modelo NM',
               marker='x', color='#2980b9', s=40)
    ax.set_xlabel('ln(Wj - Erj * t)')
    ax.set_ylabel('Produccion (kg/ha)')
    ax.set_title(f'Nelder-Mead — {nombre}\nECM={r.fun:.2e}  (ECM*={ecm_verdadero:.2e})')
    ax.legend()
plt.tight_layout()
plt.savefig('ajuste_neldermead.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: ajuste_neldermead.png]")

# Proyecciones Nelder-Mead
mejor_nm = min(res_nm.values(), key=lambda r: r.fun)
c1_nm, c2_nm, c3_nm = mejor_nm.x

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
axes = axes.flatten()
for idx, gestion in enumerate(gestiones_u):
    ax = axes[idx]
    for _, fila in df_base[df_base['Gestion'] == gestion].iterrows():
        y = produccion(t100, fila['Wj'], fila['Erj'],
                       fila['Pi'], c1_nm, c2_nm, c3_nm)
        ax.plot(t100, y, 'o-', color=COLORES_NIVEL[fila['Nivel_label']],
                label=fila['Nivel_label'], markersize=2, linewidth=1.5)
    ax.set_title(NOMBRE_LARGO.get(gestion, gestion))
    ax.set_xlabel('Anos')
    ax.set_ylabel('Produccion (kg/ha/ano)')
    hd, lb = ax.get_legend_handles_labels()
    ax.legend(dict(zip(lb, hd)).values(), dict(zip(lb, hd)).keys(), fontsize=8)
for j in range(idx+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Proyeccion a 100 anos — Nelder-Mead (mejor caso)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('proyeccion_100a_neldermead.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: proyeccion_100a_neldermead.png]")


In [ ]:
# ── BLOQUE 6: Evolución Diferencial ─────────────────────────────────────────
LIMITES = [(-100, 100), (-100, 100), (-100, 100)]
print("Ejecutando Evolucion Diferencial... (puede tardar unos segundos)")

res_ed = differential_evolution(ecm, LIMITES, seed=42, maxiter=3000,
                                 tol=1e-12, workers=1)
c1_ed, c2_ed, c3_ed = res_ed.x

print()
print("=" * 65)
print("EVOLUCION DIFERENCIAL")
print("=" * 65)
print(f"  Convergencia : {'SI' if res_ed.success else 'NO'}")
print(f"  c1 = {c1_ed:+.6f}   (verdadero: {C1_TRUE:+.4f}   error: {abs(c1_ed-C1_TRUE):.6f})")
print(f"  c2 = {c2_ed:+.6f}   (verdadero: {C2_TRUE:+.4f}   error: {abs(c2_ed-C2_TRUE):.6f})")
print(f"  c3 = {c3_ed:+.6f}   (verdadero: {C3_TRUE:+.4f}   error: {abs(c3_ed-C3_TRUE):.6f})")
print(f"  ECM        = {res_ed.fun:.2e}")
print(f"  ECM*       = {ecm_verdadero:.2e}  (cota inferior: debe ser ~0)")

P_pred_ed = Pi_all * (c1_ed + c2_ed * ln_x_all + c3_ed * ln_x_all**2)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ln_x_all, P_obs_all, label='P_obs (exacta)',
           color='#555', alpha=0.4, s=25)
ax.scatter(ln_x_all, P_pred_ed, label='Modelo Ev. Dif.',
           marker='x', color='#27ae60', s=40)
ax.set_xlabel('ln(Wj - Erj * t)')
ax.set_ylabel('Produccion (kg/ha)')
ax.set_title(f'Ajuste — Evolucion Diferencial   ECM={res_ed.fun:.2e}')
ax.legend()
plt.tight_layout()
plt.savefig('ajuste_evolucion_diferencial.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: ajuste_evolucion_diferencial.png]")

# Proyecciones a 100 años (Evolución Diferencial)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
axes = axes.flatten()
for idx, gestion in enumerate(gestiones_u):
    ax = axes[idx]
    for _, fila in df_base[df_base['Gestion'] == gestion].iterrows():
        y = produccion(t100, fila['Wj'], fila['Erj'],
                       fila['Pi'], c1_ed, c2_ed, c3_ed)
        ax.plot(t100, y, 'o-', color=COLORES_NIVEL[fila['Nivel_label']],
                label=fila['Nivel_label'], markersize=2, linewidth=1.5)
    ax.set_title(NOMBRE_LARGO.get(gestion, gestion))
    ax.set_xlabel('Anos')
    ax.set_ylabel('Produccion (kg/ha/ano)')
    hd, lb = ax.get_legend_handles_labels()
    ax.legend(dict(zip(lb, hd)).values(), dict(zip(lb, hd)).keys(), fontsize=8)
for j in range(idx+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Proyeccion a 100 anos — Evolucion Diferencial', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('proyeccion_100a_evolucion_diferencial.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: proyeccion_100a_evolucion_diferencial.png]")

# Proyecciones hasta el mínimo de producción
T_MAX  = 50_000
t_larg = np.arange(0, T_MAX + 1)
ncols2 = 4
nrows2 = int(np.ceil(len(df_base) / 4))

fig, axes2 = plt.subplots(nrows2, ncols2, figsize=(ncols2*5, nrows2*4))
axes2 = axes2.flatten()
resumen_min = []

for idx, (_, fila) in enumerate(df_base.iterrows()):
    ax = axes2[idx]
    y  = produccion(t_larg, fila['Wj'], fila['Erj'],
                    fila['Pi'], c1_ed, c2_ed, c3_ed)
    t_min = T_MAX
    for j in range(1, len(y)):
        if not np.isnan(y[j-1]) and not np.isnan(y[j]) and y[j] > y[j-1]:
            t_min = j; break
        elif np.isnan(y[j]):
            t_min = j; break
    label_min = f'Ano {t_min:,}' if t_min < T_MAX else '>Horizonte'
    color = COLORES_NIVEL[fila['Nivel_label']]
    ax.plot(t_larg[:t_min], y[:t_min], '-', color=color, linewidth=1.5)
    if t_min < T_MAX:
        ax.axvline(t_min, color='red', linestyle='--', lw=1,
                   label=f'Min: {label_min}')
    ax.set_title(f"{fila['Gestion']} — {fila['Nivel_label']}", fontsize=9)
    ax.set_xlabel('Anos', fontsize=8)
    ax.set_ylabel('Prod. (kg/ha)', fontsize=8)
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(
        ticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
    resumen_min.append({
        'Gestion':    fila['Gestion'],
        'Nivel':      fila['Nivel_label'],
        'Ano minimo': label_min,
    })

for j in range(idx+1, len(axes2)):
    axes2[j].set_visible(False)
plt.suptitle('Proyeccion hasta minimo — Evolucion Diferencial', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('proyeccion_hasta_minimo_ed.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: proyeccion_hasta_minimo_ed.png]")
print()
print(pd.DataFrame(resumen_min).to_string(index=False))


In [ ]:
# ── BLOQUE 7: Comparativa de métodos y exportación ──────────────────────────
comparativa = [
    {
        'Metodo':  'L-BFGS-B (mejor de 100)',
        'c1': c1_b, 'c2': c2_b, 'c3': c3_b, 'ECM': mejor_b['ECM'],
        'Err_c1': abs(c1_b - C1_TRUE),
        'Err_c2': abs(c2_b - C2_TRUE),
        'Err_c3': abs(c3_b - C3_TRUE),
        'Conv': f"{df_bfgs['convergio'].sum()}/{N_REAL}",
    },
    {
        'Metodo':  'Nelder-Mead Caso 1',
        'c1': res_nm['Caso 1'].x[0], 'c2': res_nm['Caso 1'].x[1],
        'c3': res_nm['Caso 1'].x[2], 'ECM': res_nm['Caso 1'].fun,
        'Err_c1': abs(res_nm['Caso 1'].x[0] - C1_TRUE),
        'Err_c2': abs(res_nm['Caso 1'].x[1] - C2_TRUE),
        'Err_c3': abs(res_nm['Caso 1'].x[2] - C3_TRUE),
        'Conv': 'Si' if res_nm['Caso 1'].success else 'No',
    },
    {
        'Metodo':  'Nelder-Mead Caso 2',
        'c1': res_nm['Caso 2'].x[0], 'c2': res_nm['Caso 2'].x[1],
        'c3': res_nm['Caso 2'].x[2], 'ECM': res_nm['Caso 2'].fun,
        'Err_c1': abs(res_nm['Caso 2'].x[0] - C1_TRUE),
        'Err_c2': abs(res_nm['Caso 2'].x[1] - C2_TRUE),
        'Err_c3': abs(res_nm['Caso 2'].x[2] - C3_TRUE),
        'Conv': 'Si' if res_nm['Caso 2'].success else 'No',
    },
    {
        'Metodo':  'Evolucion Diferencial',
        'c1': c1_ed, 'c2': c2_ed, 'c3': c3_ed, 'ECM': res_ed.fun,
        'Err_c1': abs(c1_ed - C1_TRUE),
        'Err_c2': abs(c2_ed - C2_TRUE),
        'Err_c3': abs(c3_ed - C3_TRUE),
        'Conv': 'Si' if res_ed.success else 'No',
    },
]
df_comp = pd.DataFrame(comparativa)

print(f"ECM con parametros VERDADEROS (cota inferior): {ecm_verdadero:.2e}")
print()
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
print(df_comp[['Metodo', 'c1', 'c2', 'c3', 'ECM',
               'Err_c1', 'Err_c2', 'Err_c3', 'Conv']].to_string(index=False))
pd.reset_option('display.float_format')

# Gráfica comparativa
fig, axes_c = plt.subplots(1, 2, figsize=(13, 5))
cols_m = ['#e74c3c', '#3498db', '#9b59b6', '#27ae60']
bars = axes_c[0].bar(df_comp['Metodo'], df_comp['ECM'],
                     color=cols_m, alpha=0.85, edgecolor='white')
axes_c[0].axhline(ecm_verdadero, color='black', linestyle='--', lw=1.5,
                  label=f'ECM* = {ecm_verdadero:.2e}')
for bar, val in zip(bars, df_comp['ECM']):
    axes_c[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.005,
                   f'{val:.2e}', ha='center', va='bottom', fontsize=8)
axes_c[0].set_ylabel('ECM (kg/ha)²')
axes_c[0].set_title('ECM por metodo de optimizacion')
axes_c[0].set_xticklabels(df_comp['Metodo'], rotation=20, ha='right')
axes_c[0].legend()

x_pos = np.arange(len(df_comp))
w = 0.25
for i, (col_err, label, color) in enumerate([
    ('Err_c1', '|error c1|', '#e74c3c'),
    ('Err_c2', '|error c2|', '#3498db'),
    ('Err_c3', '|error c3|', '#27ae60'),
]):
    axes_c[1].bar(x_pos + i*w, df_comp[col_err], w,
                  label=label, color=color, alpha=0.8)
axes_c[1].set_xticks(x_pos + w)
axes_c[1].set_xticklabels(df_comp['Metodo'], rotation=20, ha='right')
axes_c[1].set_ylabel('Error absoluto en el parametro')
axes_c[1].set_title('Precision de los parametros recuperados')
axes_c[1].legend()

plt.tight_layout()
plt.savefig('comparativa_metodos.png', bbox_inches='tight')
plt.show()
print("[Insertar en el TFG: comparativa_metodos.png]")

# Exportación a Excel
with pd.ExcelWriter('resultados_TFG.xlsx', engine='openpyxl') as writer:
    df_fit.to_excel(writer,  sheet_name='Datos_ajuste',        index=False)
    df_comp.to_excel(writer, sheet_name='Comparativa_metodos', index=False)
    pd.DataFrame(resumen_min).to_excel(writer,
                 sheet_name='Anios_minimo', index=False)
    df_suelo.to_excel(writer, sheet_name='Suelos_teoricos',    index=False)

mejor = df_comp.loc[df_comp['ECM'].idxmin()]
print()
print("=" * 65)
print("RESUMEN FINAL")
print("=" * 65)
print(f"  Parametros verdaderos (Rodriguez Sousa et al., 2019):")
print(f"    c1* = {C1_TRUE:+.4f}   c2* = {C2_TRUE:+.4f}   c3* = {C3_TRUE:+.4f}")
print()
print(f"  ECM* (cota inferior, sin ruido) = {ecm_verdadero:.2e}")
print()
print(f"  Metodo recomendado: Evolucion Diferencial")
print(f"    c1 = {c1_ed:+.6f}  (error: {abs(c1_ed-C1_TRUE):.2e})")
print(f"    c2 = {c2_ed:+.6f}  (error: {abs(c2_ed-C2_TRUE):.2e})")
print(f"    c3 = {c3_ed:+.6f}  (error: {abs(c3_ed-C3_TRUE):.2e})")
print(f"    ECM = {res_ed.fun:.2e}")
print()
print("Archivos generados:")
for f in ['grafica_datos_teorico.png', 'grafica_datos_campo.png',
          'ajuste_lbfgsb.png', 'proyeccion_100a_lbfgsb.png',
          'ajuste_neldermead.png', 'proyeccion_100a_neldermead.png',
          'ajuste_evolucion_diferencial.png',
          'proyeccion_100a_evolucion_diferencial.png',
          'proyeccion_hasta_minimo_ed.png',
          'comparativa_metodos.png', 'resultados_TFG.xlsx']:
    print(f"  {f}")
